# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [3]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

# Target Distribution 

In [5]:
# Distribution of target
print(fraud['fraud'].value_counts(normalize=True))
print(fraud['fraud'].value_counts())

# Yes: credit card fraud is strongly imbalanced in typical datasets,
# with frauds as a small minority class. [web:34][web:35][web:36][web:38]


fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64
fraud
0.0    912597
1.0     87403
Name: count, dtype: int64


# split + train + evaluate 

In [6]:
def train_eval_logreg(X, y, desc=""):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\n====", desc, "====")
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))
    return model


# Features/Target

In [9]:
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']


# Logistic Regression

In [10]:
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']


# Oversampling

In [12]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

ros = RandomOverSampler(random_state=42)
X_over, y_over = ros.fit_resample(X, y)

# Check new distribution
print("\nOversampled target distribution:\n", y_over.value_counts())

model_over = train_eval_logreg(X_over, y_over, desc="Random Oversampling")
# Typically improves recall/F1 for fraud at the cost of some accuracy. [web:43]



Oversampled target distribution:
 fraud
0.0    912597
1.0    912597
Name: count, dtype: int64


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



==== Random Oversampling ====
Confusion matrix:
 [[170387  12133]
 [  9133 173386]]

Classification report:
               precision    recall  f1-score   support

         0.0     0.9491    0.9335    0.9413    182520
         1.0     0.9346    0.9500    0.9422    182519

    accuracy                         0.9417    365039
   macro avg     0.9419    0.9417    0.9417    365039
weighted avg     0.9419    0.9417    0.9417    365039



# Undersampling

In [14]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X, y)

# Check new distribution
print("\nUndersampled target distribution:\n", y_under.value_counts())

model_under = train_eval_logreg(X_under, y_under, desc="Random Undersampling")
# Often increases recall on fraud but may reduce overall accuracy
# because many majority-class samples are discarded. [web:43]



Undersampled target distribution:
 fraud
0.0    87403
1.0    87403
Name: count, dtype: int64


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



==== Random Undersampling ====
Confusion matrix:
 [[16343  1138]
 [  874 16607]]

Classification report:
               precision    recall  f1-score   support

         0.0     0.9492    0.9349    0.9420     17481
         1.0     0.9359    0.9500    0.9429     17481

    accuracy                         0.9425     34962
   macro avg     0.9426    0.9425    0.9424     34962
weighted avg     0.9426    0.9425    0.9424     34962



# Smote 

In [16]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_smote, y_smote = sm.fit_resample(X, y)

print("\nSMOTE target distribution:\n", y_smote.value_counts())

model_smote = train_eval_logreg(X_smote, y_smote, desc="SMOTE Oversampling")
# SMOTE usually gives a better trade-off than naive oversampling
# by creating synthetic minority examples rather than duplicates. [web:40][web:43]



SMOTE target distribution:
 fraud
0.0    912597
1.0    912597
Name: count, dtype: int64


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



==== SMOTE Oversampling ====
Confusion matrix:
 [[170515  12005]
 [  9118 173401]]

Classification report:
               precision    recall  f1-score   support

         0.0     0.9492    0.9342    0.9417    182520
         1.0     0.9353    0.9500    0.9426    182519

    accuracy                         0.9421    365039
   macro avg     0.9422    0.9421    0.9421    365039
weighted avg     0.9422    0.9421    0.9421    365039

